In [1]:

import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

'0.8.0'

In [2]:
spark = create_spark_session(
    start_spark_cluster=True,
    executor_instances=20,
    executor_memory="16g",
    executor_cores=4,
)

INFO:teehr.evaluation.spark_session_utils:🚀 Creating Spark session: TEEHR Evaluation
INFO:teehr.evaluation.spark_session_utils:📦 Configuring Spark cluster with container image: None
INFO:teehr.evaluation.spark_session_utils:🔍 Initial spark namespace from ENV: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔍 Connecting to Kubernetes API: https://172.20.0.1:443
INFO:teehr.evaluation.spark_session_utils:🎯 Executor namespace: teehr-hub
INFO:teehr.evaluation.spark_session_utils:🔐 Executor service account: spark (in teehr-hub)
INFO:teehr.evaluation.spark_session_utils:🔐 Using in-cluster authentication
INFO:teehr.evaluation.spark_session_utils:🔗 Setting driver host to pod IP: 10.0.1.73
INFO:teehr.evaluation.spark_session_utils:✅ Spark cluster configuration successful!
INFO:teehr.evaluation.spark_session_utils:   - Executor instances: 20
INFO:teehr.evaluation.spark_session_utils:   - Executor memory: 16g
INFO:teehr.evaluation.spark_session_utils:   - Executor cores: 4
INFO:teehr.evaluatio

In [3]:
%%time
ev = teehr.RemoteReadWriteEvaluation(
    spark=spark
)

INFO:teehr.evaluation.evaluation:Using provided Spark session.
INFO:teehr.evaluation.evaluation:Active catalog set to iceberg.


CPU times: user 4.37 ms, sys: 0 ns, total: 4.37 ms
Wall time: 609 ms


In [4]:
# ev.table("sim_joined_timeseries").to_sdf().select("secondary_configuration_name").distinct().show()

In [5]:
from teehr import DeterministicMetrics as dm
from teehr import Signatures as s

metrics = [
        s.Count(),
        # s.Minimum(input_field_names=["value_time"], output_field_name="min_value_time"),
        # s.Maximum(input_field_names=["value_time"], output_field_name="max_value_time"),
        s.Average(),
        dm.RelativeBias(add_epsilon=True),
        dm.NashSutcliffeEfficiency(add_epsilon=True),
        dm.KlingGuptaEfficiency(add_epsilon=True),
        dm.RootMeanStandardDeviationRatio(add_epsilon=True),
]
metric_columns = [metric.output_field_name for metric in metrics]
group_by = ["location_id", "secondary_configuration_name", "secondary_variable_name", "unit_name"]

In [6]:
metrics = ev.table("sim_joined_timeseries").aggregate(
    group_by=group_by,
    metrics=metrics
).order_by(group_by).add_geometry()

INFO:teehr.evaluation.tables.generic_table:Getting table: sim_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: sim_joined_timeseries.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.sim_joined_timeseries.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.sim_joined_timeseries.
INFO:teehr.evaluation.dataframe_base:Performing the aggregation.
INFO:teehr.evaluation.dataframe_base:Setting order_by ['location_id', 'secondary_configuration_name', 'secondary_variable_name', 'unit_name'].
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: locations.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.locations.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.locations.
INFO:teehr.evaluation.tables.generic_table:Getting table: location_id_aliases.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: location_id_aliases.
INFO:teehr.evaluation.tables.base

In [7]:
# metrics.to_sdf().show()

In [8]:
table_name = "sim_metrics_by_location"

try:

    metrics.write_to(
        table_name=table_name,
        write_mode="create_or_replace" # overwrite would allow time travel after we move to prod.
    )

    properties = {
        "description": "Simulation metrics by location ID",
        "group_by": ", ".join(group_by),
        "metrics": ", ".join(metric_columns)
    }

    for key, value in properties.items():
        ev.spark.sql(f"""
            ALTER TABLE iceberg.teehr.{table_name} SET TBLPROPERTIES ('{key}' = '{value}')
        """)

except Exception as e:
    print(f"An error occurred: {e}")
finally:
    print("Finished attempting to write metrics table.")
    spark.stop()

INFO:teehr.evaluation.dataframe_base:Writing to table: sim_metrics_by_location.
INFO:teehr.evaluation.tables.generic_table:Getting table: sim_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: sim_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.sim_metrics_by_location.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.sim_metrics_by_location.
INFO:teehr.evaluation.write:Start writing to warehouse table 'sim_metrics_by_location'.
INFO:teehr.evaluation.tables.generic_table:Getting table: sim_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Initializing Table for table: sim_metrics_by_location.
INFO:teehr.evaluation.tables.base_table:Loading files from iceberg.teehr.sim_metrics_by_location.
INFO:teehr.evaluation.read:Reading files from iceberg.teehr.sim_metrics_by_location.
INFO:teehr.evaluation.write:Finished writing to warehouse table 'sim_metrics_by_location' in 68.072 seconds.


Finished attempting to write metrics table.


In [9]:
spark.stop()